In [ ]:
#imports
import pandas as pd
import numpy as np
from google.cloud import bigquery
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

PROJECT_ID = 'comp90089-507410'

client = bigquery.Client(project=PROJECT_ID)
INTERVAL = 6
MISSING = 0.5

def run_query(query):
    query_job = client.query(query)
    return query_job.to_dataframe()

# Cohort Consturction

**Cohort definition**
* Adult ICU patients with mechanical ventilation
    * Defined as first planned ICU extubation per patient with preceding invasive ventilation during the same icu stay (adapted from Zhao et al. 2021)
    * Unplanned events excluded
* First extubation
    * First planned extubation per patient, different to Zhao et al.
    * Unplanned Extubation (patient-initiated) itemid: 225468
    * Unplanned Extubation (non-patient-initiated) itemid: 225477
    * Extubation itemid: 227194
* $6$ hours of valid ( $\leq 50\%$ overall missingness) physiological data before extubation, assessed before imputation
* Exclusion Criteria:
    * Do-Not-Intubate (DNI) or Do Not Resuscitate (DNR)
    * Unplanned extubation - patient - and non patient initiated (procedure events)
* Missing status code retained as unkown

In [ ]:
counts = f"""
WITH base AS (
    SELECT i.*, ps.starttime AS `extubation_time`, d.label AS `extubation_type`
    FROM `physionet-data.mimiciv_3_1_derived.icustay_detail` AS i
    JOIN  `physionet-data.mimiciv_3_1_icu.procedureevents` AS ps
    ON i.stay_id = ps.stay_id
    JOIN  `physionet-data.mimiciv_3_1_icu.d_items` AS d
    ON ps.itemid = d.itemid
    WHERE (ps.itemid = 227194 OR ps.itemid = 225468 OR ps.itemid = 225477)
    AND i.icu_intime <= ps.starttime AND i.icu_outtime >= ps.starttime
), adult AS (
    SELECT *,
    FROM base
    WHERE admission_age >= 18
), planned_extubation AS(
    SELECT *
    FROM adult
    WHERE extubation_type = 'Extubation'
), first_extubation AS (
    SELECT *
    FROM planned_extubation QUALIFY ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY extubation_time) = 1
), vent_exists AS (
    SELECT *
    FROM first_extubation AS fe
    WHERE EXISTS (
        SELECT 1, stay_id
        FROM `physionet-data.mimiciv_3_1_derived.ventilation` AS v
        WHERE v.stay_id = fe.stay_id AND v.ventilation_status = 'InvasiveVent'
        AND v.starttime < fe.extubation_time
    )
), DNR_DNI_exclusion AS (
    SELECT ve.*, value, ce.charttime
    FROM vent_exists as ve
    LEFT JOIN `physionet-data.mimiciv_3_1_icu.chartevents` as ce
    ON ve.stay_id = ce.stay_id AND ce.charttime <= extubation_time AND ce.itemid = 223758
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ve.subject_id ORDER BY ce.charttime DESC) = 1
), DNR_DNI_excluded AS (
    SELECT * EXCEPT (value, charttime)
    FROM DNR_DNI_exclusion AS dde
    WHERE NOT (dde.value = 'DNR / DNI' OR dde.value = 'DNR' OR dde.value = 'DNI') or dde.value IS NULL
), deaths AS (
    SELECT dde.*, CASE WHEN ( 
        EXISTS(
            SELECT 1
            FROM `physionet-data.mimiciv_3_1_hosp.admissions` as a
            WHERE a.subject_id = dde.subject_id AND
            dde.extubation_time < a.deathtime AND 
            a.deathtime <= dde.extubation_time + INTERVAL 48 HOUR
        )
    ) 
    THEN 1 ELSE 0 END AS `death_within_48_hours`
    FROM DNR_DNI_excluded AS dde
), reintubation AS (
    SELECT d.*, CASE WHEN (
        EXISTS(
            SELECT 1,
            FROM  `physionet-data.mimiciv_3_1_icu.procedureevents` AS ps
            WHERE ps.subject_id = d.subject_id AND
            ps.itemid=224385 AND
            ps.starttime > d.extubation_time AND ps.starttime <= d.extubation_time + INTERVAL 48 HOUR
        )
    ) THEN 1 ELSE 0 END AS `reintubation_within_48_hours`
    FROM deaths AS d
)

SELECT 1 AS `Step`, COUNT(DISTINCT subject_id) AS `Patients`, 'base' AS `Stage` FROM base
UNION ALL
SELECT 2 AS `Step`, COUNT(DISTINCT subject_id) AS `Patients`, 'remove non-adult' AS `Stage` FROM adult
UNION ALL
SELECT 3 AS `Step`, COUNT(DISTINCT subject_id) AS `Patients`, 'remove unplanned extubation (patient and non-patient initiated)' AS `Stage` FROM planned_extubation
UNION ALL
SELECT 4 AS `Step`, COUNT(DISTINCT subject_id) AS `Patients`, 'remove non-first extubation' AS `Stage` FROM first_extubation
UNION ALL
SELECT 5 AS `Step`, COUNT(DISTINCT subject_id) AS `Patients`, 'remove patients without invasive ventilation prior to extubation' AS `Stage` FROM vent_exists
UNION ALL
SELECT 6 AS `Step`, COUNT(DISTINCT subject_id) AS `Patients`, 'remove patients with DNR/DNI orders' AS `Stage` FROM DNR_DNI_excluded
ORDER BY `Step`
"""
counts_result = run_query(counts)
counts_result

In [ ]:
cohort_query = f"""
WITH base AS (
    SELECT i.*, ps.starttime AS `extubation_time`, d.label AS `extubation_type`
    FROM `physionet-data.mimiciv_3_1_derived.icustay_detail` AS i
    JOIN  `physionet-data.mimiciv_3_1_icu.procedureevents` AS ps
    ON i.stay_id = ps.stay_id
    JOIN  `physionet-data.mimiciv_3_1_icu.d_items` AS d
    ON ps.itemid = d.itemid
    WHERE (ps.itemid = 227194 OR ps.itemid = 225468 OR ps.itemid = 225477)
    AND i.icu_intime <= ps.starttime AND i.icu_outtime >= ps.starttime
), adult AS (
    SELECT *,
    FROM base
    WHERE admission_age >= 18
), planned_extubation AS(
    SELECT *
    FROM adult
    WHERE extubation_type = 'Extubation'
), first_extubation AS (
    SELECT *
    FROM planned_extubation QUALIFY ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY extubation_time) = 1
), vent_exists AS (
    SELECT *
    FROM first_extubation AS fe
    WHERE EXISTS (
        SELECT 1, stay_id
        FROM `physionet-data.mimiciv_3_1_derived.ventilation` AS v
        WHERE v.stay_id = fe.stay_id AND v.ventilation_status = 'InvasiveVent'
        AND v.starttime < fe.extubation_time
    )
), DNR_DNI_exclusion AS (
    SELECT ve.*, value, ce.charttime
    FROM vent_exists as ve
    LEFT JOIN `physionet-data.mimiciv_3_1_icu.chartevents` as ce
    ON ve.stay_id = ce.stay_id AND ce.charttime <= extubation_time AND ce.itemid = 223758
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ve.subject_id ORDER BY ce.charttime DESC) = 1
), DNR_DNI_excluded AS (
    SELECT * EXCEPT (value, charttime)
    FROM DNR_DNI_exclusion AS dde
    WHERE NOT (dde.value = 'DNR / DNI' OR dde.value = 'DNR' OR dde.value = 'DNI') or dde.value IS NULL
), deaths AS (
    SELECT dde.*, CASE WHEN ( 
        EXISTS(
            SELECT 1
            FROM `physionet-data.mimiciv_3_1_hosp.admissions` as a
            WHERE a.subject_id = dde.subject_id AND
            dde.extubation_time < a.deathtime AND 
            a.deathtime <= dde.extubation_time + INTERVAL 48 HOUR
        )
    ) 
    THEN 1 ELSE 0 END AS `death_within_48_hours`
    FROM DNR_DNI_excluded AS dde
), reintubation AS (
    SELECT d.*, CASE WHEN (
        EXISTS(
            SELECT 1,
            FROM  `physionet-data.mimiciv_3_1_icu.procedureevents` AS ps
            WHERE ps.subject_id = d.subject_id AND
            ps.itemid=224385 AND
            ps.starttime > d.extubation_time AND ps.starttime <= d.extubation_time + INTERVAL 48 HOUR
        )
    ) THEN 1 ELSE 0 END AS `reintubation_within_48_hours`
    FROM deaths AS d
)

SELECT * FROM reintubation

"""
cohort = run_query(cohort_query)

In [ ]:
# Vitals
query = f"""
WITH base AS (
    SELECT i.*, ps.starttime AS `extubation_time`, d.label AS `extubation_type`
    FROM `physionet-data.mimiciv_3_1_derived.icustay_detail` AS i
    JOIN  `physionet-data.mimiciv_3_1_icu.procedureevents` AS ps
    ON i.stay_id = ps.stay_id
    JOIN  `physionet-data.mimiciv_3_1_icu.d_items` AS d
    ON ps.itemid = d.itemid
    WHERE (ps.itemid = 227194 OR ps.itemid = 225468 OR ps.itemid = 225477)
    AND i.icu_intime <= ps.starttime AND i.icu_outtime >= ps.starttime
), adult AS (
    SELECT *,
    FROM base
    WHERE admission_age >= 18
), planned_extubation AS(
    SELECT *
    FROM adult
    WHERE extubation_type = 'Extubation'
), first_extubation AS (
    SELECT *
    FROM planned_extubation QUALIFY ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY extubation_time) = 1
), vent_exists AS (
    SELECT *
    FROM first_extubation AS fe
    WHERE EXISTS (
        SELECT 1, stay_id
        FROM `physionet-data.mimiciv_3_1_derived.ventilation` AS v
        WHERE v.stay_id = fe.stay_id AND v.ventilation_status = 'InvasiveVent'
        AND v.starttime < fe.extubation_time
    )
), DNR_DNI_exclusion AS (
    SELECT ve.*, value, ce.charttime
    FROM vent_exists as ve
    LEFT JOIN `physionet-data.mimiciv_3_1_icu.chartevents` as ce
    ON ve.stay_id = ce.stay_id AND ce.charttime <= extubation_time AND ce.itemid = 223758
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ve.subject_id ORDER BY ce.charttime DESC) = 1
), DNR_DNI_excluded AS (
    SELECT * EXCEPT (value, charttime)
    FROM DNR_DNI_exclusion AS dde
    WHERE NOT (dde.value = 'DNR / DNI' OR dde.value = 'DNR' OR dde.value = 'DNI') or dde.value IS NULL
), vital_signs AS (
    SELECT dde.subject_id, dde.stay_id, dde.hadm_id, ce.charttime, dde.extubation_time,
    AVG(CASE WHEN ce.itemid = 220045 AND ce.valuenum > 0 AND ce.valuenum < 300 THEN ce.valuenum END) AS heart_rate,
    AVG(CASE WHEN ce.itemid IN (220052, 220181, 225312) AND ce.valuenum > 0 AND ce.valuenum < 300 THEN ce.valuenum END) AS mbp,
    AVG(CASE WHEN ce.itemid IN (220210, 224690) AND ce.valuenum > 0 AND ce.valuenum < 70 THEN ce.valuenum END) AS resp_rate,
    ROUND(AVG(CASE WHEN ce.itemid = 223761 AND ce.valuenum > 70 AND ce.valuenum < 120 THEN (ce.valuenum - 32) / 1.8 WHEN ce.itemid = 223762 AND ce.valuenum > 10 AND ce.valuenum < 50 THEN ce.valuenum END), 2) AS temperature,
    AVG(CASE WHEN ce.itemid = 220277 AND ce.valuenum > 0 AND ce.valuenum <= 100 THEN ce.valuenum END) AS spo2
    FROM DNR_DNI_excluded AS dde
    JOIN `physionet-data.mimiciv_3_1_icu.chartevents` AS ce
    ON dde.stay_id = ce.stay_id AND dde.subject_id = ce.subject_id
    WHERE ce.itemid IN (220045, 220052, 220181, 225312, 220210, 224690, 223761, 223762, 220277)
    AND ce.charttime >= dde.extubation_time - INTERVAL {INTERVAL} HOUR AND ce.charttime < dde.extubation_time
    AND ce.storetime <= dde.extubation_time
    GROUP BY dde.subject_id, dde.stay_id, dde.hadm_id, ce.charttime, dde.extubation_time
)
SELECT * FROM vital_signs
"""
vitals = run_query(query)

In [ ]:
# ventilator settings
query = f"""
WITH base AS (
    SELECT i.*, ps.starttime AS `extubation_time`, d.label AS `extubation_type`
    FROM `physionet-data.mimiciv_3_1_derived.icustay_detail` AS i
    JOIN  `physionet-data.mimiciv_3_1_icu.procedureevents` AS ps
    ON i.stay_id = ps.stay_id
    JOIN  `physionet-data.mimiciv_3_1_icu.d_items` AS d
    ON ps.itemid = d.itemid
    WHERE (ps.itemid = 227194 OR ps.itemid = 225468 OR ps.itemid = 225477)
    AND i.icu_intime <= ps.starttime AND i.icu_outtime >= ps.starttime
), adult AS (
    SELECT *,
    FROM base
    WHERE admission_age >= 18
), planned_extubation AS(
    SELECT *
    FROM adult
    WHERE extubation_type = 'Extubation'
), first_extubation AS (
    SELECT *
    FROM planned_extubation QUALIFY ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY extubation_time) = 1
), vent_exists AS (
    SELECT *
    FROM first_extubation AS fe
    WHERE EXISTS (
        SELECT 1, stay_id
        FROM `physionet-data.mimiciv_3_1_derived.ventilation` AS v
        WHERE v.stay_id = fe.stay_id AND v.ventilation_status = 'InvasiveVent'
        AND v.starttime < fe.extubation_time
    )
), DNR_DNI_exclusion AS (
    SELECT ve.*, value, ce.charttime
    FROM vent_exists as ve
    LEFT JOIN `physionet-data.mimiciv_3_1_icu.chartevents` as ce
    ON ve.stay_id = ce.stay_id AND ce.charttime <= extubation_time AND ce.itemid = 223758
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ve.subject_id ORDER BY ce.charttime DESC) = 1
), DNR_DNI_excluded AS (
    SELECT * EXCEPT (value, charttime)
    FROM DNR_DNI_exclusion AS dde
    WHERE NOT (dde.value = 'DNR / DNI' OR dde.value = 'DNR' OR dde.value = 'DNI') or dde.value IS NULL
), ventilator AS (
    SELECT dde.subject_id, dde.stay_id, dde.hadm_id, ce.charttime, dde.extubation_time,
    MAX(CASE WHEN ce.itemid = 224689 THEN ce.valuenum END) AS respiratory_rate_spontaneous,
    MAX(CASE WHEN ce.itemid = 224686 THEN ce.valuenum END) AS tidal_volume_spontaneous
    FROM `physionet-data.mimiciv_3_1_icu.chartevents` AS ce
    JOIN DNR_DNI_excluded AS dde
    ON ce.stay_id = dde.stay_id AND ce.subject_id = dde.subject_id
    WHERE ce.itemid IN (224689, 224686) AND ce.valuenum > 0
    AND ce.charttime >= dde.extubation_time - INTERVAL {INTERVAL} HOUR AND ce.charttime < dde.extubation_time
    AND ce.storetime <= dde.extubation_time
    GROUP BY dde.subject_id, dde.stay_id, dde.hadm_id, ce.charttime, dde.extubation_time
)
SELECT * FROM ventilator
"""
ventilator = run_query(query)

In [ ]:
# blood gases
query = f"""
WITH base AS (
    SELECT i.*, ps.starttime AS `extubation_time`, d.label AS `extubation_type`
    FROM `physionet-data.mimiciv_3_1_derived.icustay_detail` AS i
    JOIN  `physionet-data.mimiciv_3_1_icu.procedureevents` AS ps
    ON i.stay_id = ps.stay_id
    JOIN  `physionet-data.mimiciv_3_1_icu.d_items` AS d
    ON ps.itemid = d.itemid
    WHERE (ps.itemid = 227194 OR ps.itemid = 225468 OR ps.itemid = 225477)
    AND i.icu_intime <= ps.starttime AND i.icu_outtime >= ps.starttime
), adult AS (
    SELECT *,
    FROM base
    WHERE admission_age >= 18
), planned_extubation AS(
    SELECT *
    FROM adult
    WHERE extubation_type = 'Extubation'
), first_extubation AS (
    SELECT *
    FROM planned_extubation QUALIFY ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY extubation_time) = 1
), vent_exists AS (
    SELECT *
    FROM first_extubation AS fe
    WHERE EXISTS (
        SELECT 1, stay_id
        FROM `physionet-data.mimiciv_3_1_derived.ventilation` AS v
        WHERE v.stay_id = fe.stay_id AND v.ventilation_status = 'InvasiveVent'
        AND v.starttime < fe.extubation_time
    )
), DNR_DNI_exclusion AS (
    SELECT ve.*, value, ce.charttime
    FROM vent_exists as ve
    LEFT JOIN `physionet-data.mimiciv_3_1_icu.chartevents` as ce
    ON ve.stay_id = ce.stay_id AND ce.charttime <= extubation_time AND ce.itemid = 223758
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ve.subject_id ORDER BY ce.charttime DESC) = 1
), DNR_DNI_excluded AS (
    SELECT * EXCEPT (value, charttime)
    FROM DNR_DNI_exclusion AS dde
    WHERE NOT (dde.value = 'DNR / DNI' OR dde.value = 'DNR' OR dde.value = 'DNI') or dde.value IS NULL
),  blood_gas AS (
    SELECT dde.subject_id, dde.stay_id, dde.hadm_id, dde.extubation_time, l.specimen_id, l.valuenum AS po2, l.charttime, l.storetime
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` AS l
    JOIN DNR_DNI_excluded AS dde
    ON l.hadm_id = dde.hadm_id AND l.subject_id = dde.subject_id
    WHERE l.itemid = 50821 AND l.valuenum > 0
    AND l.charttime >= dde.extubation_time - INTERVAL 6 HOUR AND l.charttime < dde.extubation_time AND l.storetime <= dde.extubation_time
    AND EXISTS (
        SELECT 1 FROM `physionet-data.mimiciv_3_1_hosp.labevents` AS s
        WHERE s.subject_id = l.subject_id AND s.specimen_id = l.specimen_id AND s.itemid = 52033 AND s.value = 'ART.'
    )
), fio2 AS (
    SELECT dde.subject_id, dde.stay_id, dde.hadm_id, dde.extubation_time, l.specimen_id, CASE WHEN l.valuenum > 0.2 AND l.valuenum <= 1 THEN l.valuenum WHEN l.valuenum > 20 AND l.valuenum <= 100 THEN l.valuenum / 100 ELSE NULL END AS fio2, l.charttime, l.storetime
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` AS l
    JOIN DNR_DNI_excluded AS dde
    ON l.hadm_id = dde.hadm_id AND l.subject_id = dde.subject_id
    WHERE l.itemid = 50816
    AND l.charttime >= dde.extubation_time - INTERVAL 6 HOUR AND l.charttime < dde.extubation_time AND l.storetime <= dde.extubation_time
    AND EXISTS (
        SELECT 1 FROM `physionet-data.mimiciv_3_1_hosp.labevents` AS s
        WHERE s.subject_id = l.subject_id AND s.specimen_id = l.specimen_id AND s.itemid = 52033 AND s.value = 'ART.'
    )
)
SELECT b.*, f.fio2, f.storetime AS fio2_storetime, SAFE_DIVIDE(b.po2, f.fio2) AS pao2fio2ratio
FROM blood_gas AS b
LEFT JOIN fio2 AS f
ON b.subject_id = f.subject_id AND b.hadm_id = f.hadm_id AND b.stay_id = f.stay_id AND b.specimen_id = f.specimen_id
"""
blood_gas = run_query(query)

In [ ]:
# Convert all datetime columns to datetime
cohort['extubation_time'] = pd.to_datetime(cohort['extubation_time'])
cohort['admittime'] = pd.to_datetime(cohort['admittime'])
cohort['dischtime'] = pd.to_datetime(cohort['dischtime'])
cohort['icu_intime'] = pd.to_datetime(cohort['icu_intime'])
cohort['icu_outtime'] = pd.to_datetime(cohort['icu_outtime'])
cohort['combined_outcome'] = cohort['death_within_48_hours'] | cohort['reintubation_within_48_hours']
cohort['hospital_observed_48h'] = (cohort['dischtime'] - cohort['extubation_time']) >= pd.Timedelta(hours=48)

vitals['charttime'] = pd.to_datetime(vitals['charttime'])
vitals['extubation_time'] = pd.to_datetime(vitals['extubation_time'])

ventilator['charttime'] = pd.to_datetime(ventilator['charttime'])
ventilator['extubation_time'] = pd.to_datetime(ventilator['extubation_time'])

blood_gas['charttime'] = pd.to_datetime(blood_gas['charttime'])
blood_gas['extubation_time'] = pd.to_datetime(blood_gas['extubation_time'])

In [ ]:
# Assign bins
vitals['bin'] = np.floor(6 - (vitals['extubation_time'] - vitals['charttime']).dt.total_seconds() / 3600).astype(int)
ventilator['bin'] = np.floor(6 - (ventilator['extubation_time'] - ventilator['charttime']).dt.total_seconds() / 3600).astype(int)
blood_gas['bin'] = np.floor(6 - (blood_gas['extubation_time'] - blood_gas['charttime']).dt.total_seconds() / 3600).astype(int)

In [ ]:
id_features = ['subject_id', 'hadm_id', 'stay_id']
bin_col = ['bin']
vitals_features = [col for col in vitals.columns if col not in id_features and col not in bin_col]
ventilator_features = [col for col in ventilator.columns if col not in id_features and col not in bin_col]
blood_gas_features = [col for col in blood_gas.columns if col not in id_features and col not in bin_col]
combined_features = vitals_features + ventilator_features + blood_gas_features

In [ ]:
# Aggregate mean, standard deviation for each bin
vitals_summary_table = vitals[id_features  + vitals_features + bin_col].groupby(['subject_id', 'stay_id', 'bin']).mean().drop('hadm_id', axis=1)
ventilator_summary_table = ventilator[id_features + ventilator_features + bin_col].groupby(['subject_id', 'stay_id', 'bin']).mean().drop('hadm_id', axis=1)
blood_gas_summary_table = blood_gas[id_features + blood_gas_features + bin_col].groupby(['subject_id', 'stay_id', 'bin']).mean().drop('hadm_id', axis=1)

In [ ]:
id_cols = ['subject_id', 'stay_id', 'hadm_id']
outcome_info_cols = ['dod', 'hospital_expire_flag']
future_info_cols = ['dischtime', 'icu_outtime', 'los_hospital', 'los_icu']
misc = ['admittime', 'icu_intime', 'extubation_time', 'hospstay_seq', 'first_hosp_stay','icustay_seq','first_icu_stay','extubation_type']
outcomes = ['death_within_48_hours', 'reintubation_within_48_hours', 'combined_outcome'] # stored as separate outcomes
secondary_label = ['hospital_observed_48h']

In [ ]:
combined_table = pd.merge(vitals_summary_table, ventilator_summary_table, on=['subject_id', 'stay_id', 'bin'], how='outer', suffixes=('_vitals', '_ventilator'))
combined_table = pd.merge(combined_table, blood_gas_summary_table, on=['subject_id', 'stay_id', 'bin'], how='outer', suffixes=('', '_bg'))

keys = ['subject_id', 'stay_id']
grid = cohort[keys].drop_duplicates().merge(pd.DataFrame({'bin': range(6)}), how='cross')
combined_table = combined_table.reindex(pd.MultiIndex.from_frame(grid)).sort_index()

# This calculates the proportion of missing values for each feature across the entire dataset
raw_feature_missingness = combined_table.isna().mean()
# Overall Missingness per patient   
patient_missingness = combined_table.isna().mean(axis=1).groupby(level=keys).mean()
valid_subjects = combined_table.loc[combined_table.index.droplevel('bin').isin(patient_missingness[patient_missingness <= MISSING].index)].copy()

In [ ]:
print("Missingness report feature:\n")
display(raw_feature_missingness)
print("Number of patients with >50% overall missingness: ", len(patient_missingness[patient_missingness > 0.5]))


In [ ]:
combined_ff = combined_table.groupby(['subject_id', 'stay_id']).ffill()
combined_ff['RSBI'] = combined_ff['respiratory_rate_spontaneous'] / (combined_ff['tidal_volume_spontaneous'].where(combined_ff['tidal_volume_spontaneous'] > 0)/1000)
wide = combined_ff.unstack('bin')
wide.columns = [
    f'{feature}_bin_{num}' for feature, num in wide.columns
]
full_final = pd.merge(cohort, wide, on=['subject_id', 'stay_id'], how='inner')
clean_full_dataset = full_final.drop(outcome_info_cols + future_info_cols + misc, axis=1)
clean_full_dataset

In [ ]:
# Forward fill valid subjects
valid_subjects = valid_subjects.groupby(['subject_id', 'stay_id']).ffill()
valid_subjects['RSBI'] = valid_subjects['respiratory_rate_spontaneous'] / (valid_subjects['tidal_volume_spontaneous'].where(valid_subjects['tidal_volume_spontaneous'] > 0)/1000)

wide = valid_subjects.unstack('bin')
wide.columns = [
    f'{feature}_bin_{num}' for feature, num in wide.columns
]
final_dataset = pd.merge(cohort, wide, on=['subject_id', 'stay_id'], how='inner')
clean_filtered_dataset = final_dataset.drop(outcome_info_cols + future_info_cols + misc, axis=1)
clean_filtered_dataset

In [ ]:
# Please use this as feature list for model training
feature_list = [col for col in combined_table.columns if col not in secondary_label and col not in misc and col not in future_info_cols and col not in id_cols and col not in outcome_info_cols]
# Use combined outcome for outcome
outcome_col = 'combined_outcome'


train_set_filtered, test_set_filtered = train_test_split(
    clean_filtered_dataset,
    train_size=0.8,
    random_state=42,
    stratify=clean_filtered_dataset[outcome_col]
)

train_set_full, test_set_full = train_test_split(
    clean_full_dataset,
    train_size=0.8,
    random_state=42,
    stratify=clean_full_dataset[outcome_col]
)

full_medians = train_set_full.median(numeric_only=True)
filtered_medians = train_set_filtered.median(numeric_only=True)

train_set_full.fillna(full_medians, inplace=True)
test_set_full.fillna(full_medians, inplace=True)

train_set_filtered.fillna(filtered_medians, inplace=True)
test_set_filtered.fillna(filtered_medians, inplace=True)

In [ ]:
# Cleaned full final dataset saved to CSV
clean_full_dataset.to_csv('clean_full_final.csv', index=False)
clean_filtered_dataset.to_csv('clean_final_dataset.csv', index=False)

# Cleaned train and test sets saved to CSV
train_set_filtered.to_csv('train_set_filtered.csv', index=False)
test_set_filtered.to_csv('test_set_filtered.csv', index=False)
train_set_full.to_csv('train_set_full.csv', index=False)
test_set_full.to_csv('test_set_full.csv', index=False)

In [ ]:
patient_count_prior_to_50_filter = combined_table.unstack('bin').shape[0]
patient_count_after_50_filter = valid_subjects.unstack('bin').shape[0]
removed = patient_count_prior_to_50_filter - patient_count_after_50_filter
counts_result.loc[6] = {
    'Step': 7,
    'Patients': patient_count_after_50_filter,
    'Stage': 'remove patients with less than 50% measurement data'
}
counts_result['removed'] = (counts_result['Patients'].shift(1) - counts_result['Patients']).fillna(0)
for step, patients, stage, removed in counts_result[['Step', 'Patients', 'Stage', 'removed']].itertuples(index=False):
    print(f"Step {step}: {patients} patients - {stage}, removed {removed} patients")

In [ ]:
comparison = cohort.merge(
    patient_missingness.rename('missingness').reset_index(),
    on=['subject_id', 'stay_id'],
    how='left',
    validate='one_to_one'
)

comparison['group'] = np.where(
    comparison['missingness'] <= MISSING, 'Retained', 'Excluded'
)
comparison['combined_outcome'] = (
    comparison['death_within_48_hours']
    | comparison['reintubation_within_48_hours']
)

def median_iqr(s):
    return f'{s.median():.1f} ({s.quantile(.25):.1f}–{s.quantile(.75):.1f})'

def count_pct(s):
    return f'{int(s.sum())} ({100 * s.mean():.1f}%)'

comparison['female'] = comparison['gender'].eq('F')
comparison['missingness_pct'] = comparison['missingness'] * 100

summary = comparison.groupby('group').agg(
    patients=('subject_id', 'size'),
    age_median_IQR=('admission_age', median_iqr),
    female_n_pct=('female', count_pct),
    missingness_pct_median_IQR=('missingness_pct', median_iqr),
    reintubation_n_pct=('reintubation_within_48_hours', count_pct),
    death_n_pct=('death_within_48_hours', count_pct),
    combined_outcome_n_pct=('combined_outcome', count_pct)
).reindex(['Retained', 'Excluded'])

display(summary)

In [ ]:
# Outcome report
full_outcome_report = clean_full_dataset[outcomes].sum()
full_outcome_report['overlap'] = full_outcome_report['death_within_48_hours'] + full_outcome_report['reintubation_within_48_hours'] - full_outcome_report['combined_outcome']
full_outcome_report['positive rate'] = full_outcome_report['combined_outcome'] / clean_full_dataset.shape[0]

filtered_outcome_report = clean_filtered_dataset[outcomes].sum()
filtered_outcome_report['overlap'] = filtered_outcome_report['death_within_48_hours'] + filtered_outcome_report['reintubation_within_48_hours'] - filtered_outcome_report['combined_outcome']
filtered_outcome_report['positive rate'] = filtered_outcome_report['combined_outcome'] / clean_filtered_dataset.shape[0]

rename = {
    'death_within_48_hours': 'Death within 48 hours (N=)',
    'reintubation_within_48_hours': 'Reintubation within 48 hours (N=)',
    'combined_outcome': 'Combined outcome (N=)',
    'overlap': 'Overlap (N=)',
    'positive rate': 'Positive rate'
}

full_outcome_report.rename(index=rename, inplace=True)
filtered_outcome_report.rename(index=rename, inplace=True)

print("Full dataset outcome report\n")
display(full_outcome_report)

print("\nFiltered dataset outcome report\n")
display(filtered_outcome_report)

In [ ]:
fig = plt.figure(figsize=(5, 5))
plt.bar(['Full Dataset', '50% Filtered Dataset'], 
        [clean_full_dataset['hospital_observed_48h'].sum() / clean_full_dataset.shape[0], 
         clean_filtered_dataset['hospital_observed_48h'].sum() / clean_filtered_dataset.shape[0]], width=0.3)
plt.ylabel('Percentage of Complete Observation')
plt.title('Comparison of Complete Observations')
plt.show()
print(f"Full dataset complete observation percentage: {clean_full_dataset['hospital_observed_48h'].sum() / clean_full_dataset.shape[0] * 100:.2f}%")
print(f"50% filtered dataset complete observation percentage: {clean_filtered_dataset['hospital_observed_48h'].sum() / clean_filtered_dataset.shape[0] * 100:.2f}%")